# Feature selection

XGBoost trained on each of the 10 semantic feature groups and on four feature combinations
(7, 16, 23 and 40 dimensions) under cross-user mixing with stratified 10-fold cross-validation.

Reads `../data/window_features.csv`. The results of each fold and their means are written to `results/`.

In [ ]:
import os
import time
import warnings
from collections import OrderedDict
from inspect import signature
from time import perf_counter

import numpy as np
import pandas as pd
from imblearn.over_sampling import RandomOverSampler
from sklearn.base import clone
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

DATA_PATH = os.path.join("..", "data", "window_features.csv")
OUT_DIR = "results"
os.makedirs(OUT_DIR, exist_ok=True)

SEED = 42
np.random.seed(SEED)

df = pd.read_csv(DATA_PATH)
df = df.dropna(how="all")

## Feature groups

In [ ]:
CONV_DIST = ["Convergence_distance"]
EYE_OPEN = ["Left_Eye_Openness", "Right_Eye_Openness"]
PUPIL_DIAM = ["LeftPupilDiameter", "RightPupilDiameter"]
PUPIL_POS = ["LeftPupilPosInSensorX", "LeftPupilPosInSensorY", "RightPupilPosInSensorX", "RightPupilPosInSensorY"]
GAZE_DIR_HMD = ["NrmSRLeftEyeGazeDirX", "NrmSRLeftEyeGazeDirY", "NrmSRLeftEyeGazeDirZ",
                "NrmSRRightEyeGazeDirX", "NrmSRRightEyeGazeDirY", "NrmSRRightEyeGazeDirZ"]
COMB_GAZE_ORI = ["GazeOriginLclSpc_X", "GazeOriginLclSpc_Y", "GazeOriginLclSpc_Z",
                 "GazeOriginWrldSpc_X", "GazeOriginWrldSpc_Y", "GazeOriginWrldSpc_Z"]
COMB_GAZE_DIR = ["GazeDirectionLclSpc_X", "GazeDirectionLclSpc_Y", "GazeDirectionLclSpc_Z",
                 "GazeDirectionWrldSpc_X", "GazeDirectionWrldSpc_Y", "GazeDirectionWrldSpc_Z"]
EYE_ORI = ["NrmLeftEyeOriginX", "NrmLeftEyeOriginY", "NrmLeftEyeOriginZ",
           "NrmRightEyeOriginX", "NrmRightEyeOriginY", "NrmRightEyeOriginZ"]
HEAD_QROT = ["HeadQRotationX", "HeadQRotationY", "HeadQRotationZ", "HeadQRotationW"]
HEAD_EUL = ["HeadEulX", "HeadEulY", "HeadEulZ"]

exp_feature_groups = OrderedDict([
    ("Conv. Dist.", CONV_DIST),
    ("Eye Open.", EYE_OPEN),
    ("Pupil Diam.", PUPIL_DIAM),
    ("Pupil Pos.", PUPIL_POS),
    ("Gaze Dir. (HMD)", GAZE_DIR_HMD),
    ("Comb. Gaze Ori.", COMB_GAZE_ORI),
    ("Comb. Gaze Dir.", COMB_GAZE_DIR),
    ("Eye Ori.", EYE_ORI),
    ("Head QRot.", HEAD_QROT),
    ("Head Eul.", HEAD_EUL),
    ("7: Head QRot. + Head Eul.", HEAD_QROT + HEAD_EUL),
    ("16: Pupil Pos. + Eye Ori. + Comb. Gaze Ori.", PUPIL_POS + COMB_GAZE_ORI + EYE_ORI),
    ("23: Pupil Pos. + Eye Ori. + Comb. Gaze Ori. + Head QRot. + Head Eul.",
     PUPIL_POS + COMB_GAZE_ORI + EYE_ORI + HEAD_QROT + HEAD_EUL),
    ("40: All Eye and Head", CONV_DIST + EYE_OPEN + PUPIL_DIAM + PUPIL_POS + GAZE_DIR_HMD + COMB_GAZE_ORI
     + COMB_GAZE_DIR + EYE_ORI + HEAD_QROT + HEAD_EUL),
])

## Model and cross-validation
Same XGBoost setting and protocol as in `model_comparison/ml_models.ipynb`.

In [ ]:
def random_oversample(X, y, seed=SEED):
    ros = RandomOverSampler(random_state=seed)
    return ros.fit_resample(X, y)


def build_models():
    return [
        ("XGBoost", make_pipeline(
            StandardScaler(),
            XGBClassifier(n_estimators=400, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                          gamma=0.1, reg_alpha=0.1, objective="multi:softprob", eval_metric="mlogloss",
                          random_state=SEED, n_jobs=-1))),
    ]


def run_experiment(df_src, exp_name, feature_cols):
    df_local = df_src.dropna(subset=["cs"]).copy()
    y_all = df_local["cs"].astype(int).values
    X_all = df_local[feature_cols].to_numpy(dtype=float)
    dataset_name = "Prewindowed samples"
    labels = [0, 1, 2, 3]

    cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)
    results_folds, results_avg = [], []

    for model_name, model_proto in build_models():
        train_times, test_times = [], []
        fold_accuracies, fold_precisions, fold_recalls = [], [], []
        fold_f1_each_list, fold_f1_macro_list = [], []

        for fold_idx, (train_idx, test_idx) in enumerate(cv.split(X_all, y_all), 1):
            X_train, y_train = X_all[train_idx], y_all[train_idx]
            X_test, y_test = X_all[test_idx], y_all[test_idx]

            model = clone(model_proto)
            last_step_name = list(model.named_steps.keys())[-1]
            estimator = model.named_steps[last_step_name]
            X_train, y_train = random_oversample(X_train, y_train)

            classes = np.unique(y_train)
            weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
            class_weight_dict = {int(c): w for c, w in zip(classes, weights)}
            sample_weight = np.array([class_weight_dict[int(lbl)] for lbl in y_train])
            fit_kwargs = {}
            if "sample_weight" in signature(estimator.fit).parameters:
                fit_kwargs[f"{last_step_name}__sample_weight"] = sample_weight

            t_train0 = time.time()
            model.fit(X_train, y_train, **fit_kwargs)
            train_time = time.time() - t_train0
            train_times.append(train_time)

            t_inf0 = perf_counter()
            y_pred = model.predict(X_test)
            inference_time_per_sample = (perf_counter() - t_inf0) / len(X_test)
            test_times.append(inference_time_per_sample)

            acc = accuracy_score(y_test, y_pred)
            prec = precision_score(y_test, y_pred, labels=labels, average=None, zero_division=0)
            rec = recall_score(y_test, y_pred, labels=labels, average=None, zero_division=0)
            f1_each = f1_score(y_test, y_pred, labels=labels, average=None, zero_division=0)
            f1_macro = f1_score(y_test, y_pred, labels=labels, average="macro", zero_division=0)
            fold_accuracies.append(acc)
            fold_precisions.append(prec)
            fold_recalls.append(rec)
            fold_f1_each_list.append(f1_each)
            fold_f1_macro_list.append(f1_macro)

            results_folds.append({
                "experiment": exp_name, "dataset": dataset_name, "model": model_name, "fold": fold_idx,
                "accuracy": round(acc, 4),
                **{f"precision {i}": round(prec[i], 4) for i in range(4)},
                **{f"recall {i}": round(rec[i], 4) for i in range(4)},
                **{f"f1_{i}": round(f1_each[i], 4) for i in range(4)},
                "f1_macro": round(f1_macro, 4),
                "train_time_sec": round(train_time, 4),
                "inference_time_s_per_sample": round(inference_time_per_sample, 8),
            })

        avg_prec = np.mean(fold_precisions, axis=0)
        avg_rec = np.mean(fold_recalls, axis=0)
        avg_f1_each = np.mean(np.vstack(fold_f1_each_list), axis=0)
        results_avg.append({
            "experiment": exp_name, "dataset": dataset_name, "model": model_name,
            "n_features": len(feature_cols),
            "test accuracy": round(float(np.mean(fold_accuracies)), 4),
            **{f"test precision {i}": round(avg_prec[i], 4) for i in range(4)},
            **{f"test recall {i}": round(avg_rec[i], 4) for i in range(4)},
            **{f"test f1 {i}": round(avg_f1_each[i], 4) for i in range(4)},
            "test f1 macro": round(float(np.mean(fold_f1_macro_list)), 4),
            "time_train_mean_sec": round(float(np.mean(train_times)), 2),
            "mean_inf_time_s_per_sample": round(float(np.mean(test_times)), 8),
        })
        print(f"{exp_name}: Acc={np.mean(fold_accuracies):.3f}, Macro F1={np.mean(fold_f1_macro_list):.3f}")

    return pd.DataFrame(results_folds), pd.DataFrame(results_avg)

In [ ]:
all_folds, all_avg = [], []
for exp_name, fea_cols in exp_feature_groups.items():
    feature_cols = [c if c in df.columns else f"{c}__mean" for c in fea_cols]
    folds_df, avg_df = run_experiment(df, exp_name, feature_cols)
    all_folds.append(folds_df)
    all_avg.append(avg_df)

pd.concat(all_folds, ignore_index=True).to_csv(os.path.join(OUT_DIR, "feature_cv_results_folds.csv"), index=False)
pd.concat(all_avg, ignore_index=True).to_csv(os.path.join(OUT_DIR, "feature_cv_results_avg.csv"), index=False)